# Optimal Number of Clusters

We will now determine the Optimal Number of Clusters (ONC) which will then feed the Nested Clustered Optimisation (NCO) allocator
- First we denoise and detone the correlation
- Iterate over a series of possible number of clusters
- Score each iteration with the silhouette method
- Keep the k number of clusters with the best score

In [12]:
# Imports

import pandas as pd
import numpy as np
from vportfolio.params import LOOKBACK
from vportfolio import load_data, engine, denoising, clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

## Denoise and detone

In [13]:
# data

data = load_data.load_universe()

# let's take the last date of the research period

decision_date = data["rebal_dates"].iloc[-1]

# calculate the lookback window

window = engine.lookback_window(data["returns"], data["eligible"], decision_date, n=LOOKBACK)

# calculate the correlation matrix

corr = window.corr()

# remove the noise from the correlation matrix 
corr_dn = denoising.denoise_corr(corr, window.shape[0])

To detone the matrix we essentially need to remove the market factor or risk appetite from the matrix. This is done by removing the largest eigenvalue from the matrix

In [14]:
# detone a correlation matrix
# will return a dataframe with the detoned correlation matrix
def detone_corr(corr):
    eigenvals, eigenvec = np.linalg.eigh(corr) # smallest first
    lambda_1 = eigenvals[-1] # market factor: the largest eigenvalue
    v1 = eigenvec[:, -1] # and its eigenvector (last column)
    market_part = lambda_1 * np.outer(v1, v1) # the market's contribution to the matrix
    detoned = corr - market_part # remove it
    d = np.sqrt(np.diag(detoned)) # rescale so the diagonal is 1 again
    detoned = detoned / np.outer(d, d)
    return pd.DataFrame(detoned, index=corr.index, columns=corr.columns)

In [15]:
# testing the function

# 1. always true: symmetric, diagonal 1, no NaNs
rng = np.random.default_rng(0)
random_corr = pd.DataFrame(rng.normal(size=(500, 6))).corr()
result = detone_corr(random_corr)
assert np.allclose(result, result.T)
assert np.allclose(np.diag(result), 1)
assert result.notna().all().all()
# 2. known answer: pure market (all 0.6) → off-diagonals become −1/(n−1) = −0.25 with 5 assets
m = np.full((5, 5), 0.6)
np.fill_diagonal(m, 1)
result = detone_corr(pd.DataFrame(m))
off_diag = ~np.eye(5, dtype=bool)
assert np.allclose(result.values[off_diag], -0.25)
# 3. inputs matter: block structure survives (0.8 within blocks, 0.3 between)
m = np.full((6, 6), 0.3)
m[:3, :3] = 0.8
m[3:, 3:] = 0.8
np.fill_diagonal(m, 1)
result = detone_corr(pd.DataFrame(m))
within = result.values[0, 1] # inside block 1
between = result.values[0, 3] # block 1 vs block 2
assert within > between

print("detone_corr passed all tests")

detone_corr passed all tests


In [16]:
# let's now detone the correlation

corr_dt = detone_corr(corr_dn) 

## ONC

We now need to build the ONC iterator

In [19]:
# try k = 2..max_k clusters with k-means, score each grouping by the silhouette (mean ÷ std)
# Return the best grouping as a Series of cluster labels indexed by ticker
def onc(corr, max_k=10, n_init=10):
    dist = clustering.corr_to_distance(corr)
    max_k = min(max_k, len(corr) - 1) # silhouettes need fewer clusters than ETFs
    best_score, best_labels = -np.inf, None # list to store best results. -np.inf makes sure that the first grouping will beat it
    for k in range(2, max_k + 1): # start at 2 and add 1 cluster to each iteration
        for seed in range(n_init): # for each k, try n_init random starts for cluster positioning
            labels = KMeans(n_clusters=k, n_init=1, random_state=seed).fit(dist).labels_ # run the k-means optimizer for cluster centroids
            sil = silhouette_samples(dist, labels) # apply the scoring method to the k-means optimizer
            score = sil.mean() / sil.std()  # score the whole group with one number
        if score > best_score: # if this group beats best score
            best_score, best_labels = score, labels # add it to the empty series                        
    return pd.Series(best_labels, index=corr.index)

In [20]:
# function testing

# helper: correlations of fake returns where each block of assets shares one common factor
def block_corr(sizes, days=1000, seed=0):
    rng = np.random.default_rng(seed)
    blocks = []
    for size in sizes:
        factor = rng.normal(size=(days, 1)) # the block's shared movement
        blocks.append(factor + 0.3 * rng.normal(size=(days, size)))  # plus each asset's own noise
    returns = pd.DataFrame(np.hstack(blocks), columns=[f"X{i}" for i in range(sum(sizes))])
    return returns.corr()

# 1. always true: one label per ticker, between 2 and max_k clusters
rng = np.random.default_rng(0)
random_corr = pd.DataFrame(rng.normal(size=(500, 8)), columns=list("ABCDEFGH")).corr()
labels = onc(random_corr)
assert labels.index.equals(random_corr.index)
assert 2 <= labels.nunique() <= 7 # max_k is capped at 8 − 1 = 7

# 2. known answer: three blocks of 3 → 3 clusters, each block together
labels = onc(block_corr([3, 3, 3]))
assert labels.nunique() == 3
assert labels.iloc[0] == labels.iloc[1] == labels.iloc[2]
assert labels.iloc[3] == labels.iloc[4] == labels.iloc[5]
assert labels.iloc[6] == labels.iloc[7] == labels.iloc[8]

# 3. inputs matter: the structure sets the number of clusters (two blocks → 2, four blocks → 4)
assert onc(block_corr([3, 3])).nunique() == 2
assert onc(block_corr([4, 4, 4, 4])).nunique() == 4

print("onc passed all tests")

onc passed all tests


Let's check the number of clusters the ONC function produces from the denoised and detoned corr matrix

In [21]:
# apply ONC function

labels = onc(corr_dt)
labels.sort_values()

Ticker
SPY     0
QQQ     0
IWM     0
EFA     0
EEM     0
DBC     0
XLE     0
HYG     0
UUP     0
VNQ     0
SHY     1
IEF     1
EMB     1
LQD     1
TLT     1
TIP     1
EMLC    1
GLD     1
FXY     1
FXA     1
FXF     1
dtype: int32

On the window ending 2021-12-31, ONC finds two clusters on the denoised, detoned correlation: 
- a rate-sensitive group (Treasuries, investment-grade and EM bonds, gold, and the yen, franc and Australian dollar)
- a growth/commodity group (equities, REITs, high yield, commodities, energy and the dollar)

This matches the denoising result: 3 signal factors, minus the market factor removed by detoning, leaves 2. 

The dollar sits apart from the other currencies because UUP is close to their mirror image. Gold sits with the other currencies because it tends to behave like them; rise when dollar falls and vice versa.

Denoising compresses the distances, so ONC prefers this broad split over the finer groups visible in the raw tree.